In [1]:
import os, sys
sys.path.append(os.path.abspath('..'))

In [ ]:
# Library imports
import json
import time
import numpy as np
import pandas as pd
import torch
import gymnasium as gym

# File imports
from env import InventoryEnv
from stable_baselines3 import DQN
from stable_baselines3.dqn.policies import DQNPolicy, QNetwork

# Large but finite - NOT -inf. Negative values are used to mask out depleted warehouses
MASK_VALUE = -1e8

# Maximum distance between a facility and a customer
DISTANCE_NORM = 212.13


class ScaledRewardWrapper(gym.RewardWrapper):
    # Scale the reward by the maximum distance between a facility and a customer (212.13) so that the reward is in the range [0, 1] for easier learning.
    def reward(self, reward):
        return reward / DISTANCE_NORM


def _capacity_mask(obs, num_warehouses):
    # Only allow actions that correspond to warehouses with remaining capacity. This is used to create an action mask for the MaskablePPO algorithm.
    capacity = obs[..., 1::2][..., :num_warehouses]
    return capacity > 0


class MaskedQNetwork(QNetwork):
    def forward(self, obs):
        q_values = super().forward(obs)
        mask = _capacity_mask(obs, q_values.shape[-1])
        return q_values.masked_fill(~mask, MASK_VALUE)


class MaskedDQNPolicy(DQNPolicy):
    def make_q_net(self):
        net_args = self._update_features_extractor(self.net_args, features_extractor=None)
        return MaskedQNetwork(**net_args).to(self.device)


class MaskedDQN(DQN):
    
    def _sample_action(self, learning_starts):
        assert self._last_obs is not None, "self._last_obs was not set"
        if self.num_timesteps < learning_starts and not (self.use_sde and self.use_sde_at_warmup):
            mask = _capacity_mask(np.asarray(self._last_obs), self.action_space.n)
            unscaled_action = np.array([np.random.choice(np.flatnonzero(row)) for row in mask])
        else:
            unscaled_action, _ = self.predict(self._last_obs, deterministic=False)

        # discrete action space: no scaling/clipping needed (mirrors the discrete branch
        # of OffPolicyAlgorithm._sample_action)
        buffer_action = unscaled_action
        action = buffer_action
        return action, buffer_action

    def predict(self, observation, state=None, episode_start=None, deterministic=False):
        if not deterministic and np.random.rand() < self.exploration_rate:
            mask = _capacity_mask(np.asarray(observation), self.action_space.n)
            if mask.ndim == 1:
                action = np.array(np.random.choice(np.flatnonzero(mask)))
            else:
                action = np.array([np.random.choice(np.flatnonzero(row)) for row in mask])
        else:
            action, state = self.policy.predict(observation, state, episode_start, deterministic)
        return action, state

In [ ]:
%load_ext tensorboard
%tensorboard --logdir='deep_q_networks_training/Data_Training' --port 6001    #Change port if needed (6006,9009,9999)

In [ ]:
def train_dqn(num_warehouses, num_customers, capacity_distribution):

    SEED = 42
    np.random.seed(SEED)

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
    env = ScaledRewardWrapper(env)

    env.reset()

    model_DQN = MaskedDQN(MaskedDQNPolicy,
                    env, 
                    tensorboard_log=f"deep_q_networks_training/Data_Training/DQN/w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}/", 
                    verbose=0,
                    seed=SEED,  # seeds SB3's own internals (policy init, action_space sampling, replay buffer)
                )

    n_steps = num_customers * 50_000   

    start_time = time.perf_counter()
    model_DQN.learn(n_steps, reset_num_timesteps=True)
    training_time_minutes = (time.perf_counter() - start_time) / 60

    model_DQN.save(f'deep_q_networks_training/rl_models/dqn_models/dqn_model_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.zip')
    model_DQN.save_replay_buffer(f'deep_q_networks_training/rl_models/dqn_replay_buffer/replay_buffer_dqn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.zip')
    print('Saved DQN model')

    return training_time_minutes

In [ ]:
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ['uniform', 'uneven']

training_times = []  # one row per (num_warehouses, num_customers, capacity_distribution) family

for num_customers in num_customers_options:
    for num_warehouses in num_warehouses_options:
        for capacity_distribution in capacity_distribution_options:
            training_time_minutes = train_dqn(num_warehouses, num_customers, capacity_distribution)
            print(f"Trained model for {num_warehouses} warehouses, {num_customers} customers, {capacity_distribution} capacity")

            training_times.append({
                'num_warehouses': num_warehouses,
                'num_customers': num_customers,
                'capacity_distribution': capacity_distribution,
                'training_time': round(training_time_minutes, 2),
            })

info_dir = 'deep_q_networks_training'
os.makedirs(info_dir, exist_ok=True)

training_times_df = pd.DataFrame(training_times)
training_times_df.to_csv(os.path.join(info_dir, 'deep_q_networks_training_times.csv'), index=False, float_format='%.5f')